In [1]:
!pip install langchain langchain-groq langchain-core langchain-tavily langchain-community

In [2]:
import os

In [3]:
os.environ["GROQ_API_KEY"] = "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"

In [4]:
os.environ["TAVILY_API_KEY"] = "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"

In [5]:
import langchain

In [6]:
from langchain_tavily import TavilySearch

In [7]:
from langchain_groq import ChatGroq

c:\Users\esree\anaconda3\envs\tf_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from langchain_classic import hub

In [9]:
from langchain_classic.agents import create_react_agent, AgentExecutor

### LLM_Model

In [ ]:
#llm_model = ChatGroq(model = "llama-3.1-8b-instant")

In [73]:
llm_model = ChatGroq(model = "llama-3.3-70b-versatile")

In [74]:
llm_model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020129E0F1D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020129E0ED90>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### Web-Search Tool

In [75]:
search_tool = TavilySearch()

In [76]:
search_tool

TavilySearch(api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))

In [77]:
tools = [search_tool]

In [78]:
from langchain_classic import hub

In [79]:
!pip install -q langsmith

In [80]:
from langsmith import Client

In [81]:
client = Client()

In [82]:
prompt = client.pull_prompt(
                            "hwchase17/react",
                            include_model = False,
                            dangerously_pull_public_prompt = True

           )

### Create Agent

In [83]:
from langchain_classic.agents import create_react_agent

In [84]:
agent = create_react_agent(
                    llm = llm_model,
                    tools = tools,  #[search_tool]
                    prompt = prompt
)                


In [85]:
from langchain_classic.agents import AgentExecutor

In [86]:
agent_executor = AgentExecutor(
              agent = agent,
              tools = tools,
              verbose = False
)

In [87]:
agent_executor

AgentExecutor(verbose=False, agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={}, partial_variables={'tools': 'tavily_search - A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. It not only retrieves URLs and snippets, but offers advanced search depths, domain management, time range filters, and image search, this tool delivers real-time, accurate, and citation-backed results.Input should be a search query.', 'tool_names': 'tavily_search'}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:

In [88]:
agent_executor.invoke({"input":"in korea the most popular pop band"})

{'input': 'in korea the most popular pop band',
 'output': 'The most popular K-pop band in Korea is SEVENTEEN, although BTS is also extremely popular.'}

In [89]:
result = agent_executor.invoke({"input" : "who won west bengal assemble election 2026"})

In [90]:
result

{'input': 'who won west bengal assemble election 2026',
 'output': 'The BJP won the West Bengal assembly election in 2026, securing 207 seats, while the AITC won 80 seats.'}